# Busqueda del mejor modelo e hiperparametros — Airfoil Self-Noise

Tomamos el dataset Airfoil Self-Noise procesado por el ETL en S3 (`s3://data/final/...`) y ejecutamos una busqueda bayesiana de hiperparametros con **Optuna**, trackeando todo en **MLflow**.

Los modelos candidatos son **RandomForest**, **XGBoost** y **SVR** — uno por cada paradigma (bagging, boosting, kernel). La metrica de optimizacion es **R²** sobre 5-fold CV.

Al finalizar, registramos el mejor modelo como `airfoil_model_prod` con alias `champion` en el Model Registry, listo para ser servido por la API.

> Basado en el tutorial de [MLflow](https://mlflow.org/docs/latest/traditional-ml/hyperparameter-tuning-with-child-runs/notebooks/index.html).

In [1]:
import awswrangler as wr
import mlflow

%env AWS_ACCESS_KEY_ID=minio
%env AWS_SECRET_ACCESS_KEY=minio123
%env MLFLOW_S3_ENDPOINT_URL=http://localhost:9000
%env AWS_ENDPOINT_URL_S3=http://localhost:9000

env: AWS_ACCESS_KEY_ID=minio
env: AWS_SECRET_ACCESS_KEY=minio123
env: MLFLOW_S3_ENDPOINT_URL=http://localhost:9000
env: AWS_ENDPOINT_URL_S3=http://localhost:9000


In [2]:
mlflow_server = "http://localhost:5001"
mlflow.set_tracking_uri(mlflow_server)

## Carga de datos

Leemos los splits ya normalizados que dejo el DAG `process_etl_data` en S3.

In [3]:
X_train = wr.s3.read_csv("s3://data/final/train/X_train.csv")
y_train = wr.s3.read_csv("s3://data/final/train/y_train.csv")

X_test = wr.s3.read_csv("s3://data/final/test/X_test.csv")
y_test = wr.s3.read_csv("s3://data/final/test/y_test.csv")

print(f"Train: {X_train.shape} -> {y_train.shape}")
print(f"Test:  {X_test.shape} -> {y_test.shape}")

Train: (1202, 5) -> (1202, 1)
Test:  (301, 5) -> (301, 1)


## Analisis de relacion features ↔ target

Antes de entrenar, generamos dos graficos que vamos a loguear como artefactos en el run padre de MLflow:

- **Correlacion lineal** de cada feature con el target (SSPL).
- **Information gain** (mutual_info_regression) de cada feature.

Sirve para auditoria/trazabilidad y para detectar data leakage.

In [4]:
from plots import plot_correlation_with_target, plot_information_gain_with_target

correlation_plot = plot_correlation_with_target(X_train, y_train, target_col="SSPL")
information_gain_plot = plot_information_gain_with_target(X_train, y_train, target_col="SSPL")

## Setup del experimento

In [5]:
import datetime
import optuna

from mlflow.models import infer_signature
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

from mlflow_aux import get_or_create_experiment
from optuna_aux import champion_callback, objective, build_model_from_params

optuna.logging.set_verbosity(optuna.logging.ERROR)

experiment_id = get_or_create_experiment("Airfoil Self-Noise")
print(f"experiment_id: {experiment_id}")

run_name_parent = "best_hyperparam_" + datetime.datetime.today().strftime('%Y/%m/%d-%H:%M:%S')

experiment_id: 1


## Busqueda con Optuna

Lanzamos un estudio bayesiano sobre los 3 regresores (RandomForest, XGBoost, SVR) con sus respectivos espacios de hiperparametros. Cada trial se loguea como un run hijo del run padre. Maximizamos R² (5-fold CV).

> Bajar `n_trials` si el entorno es lento. 50 trials suelen alcanzar para converger en este dataset.

In [6]:
N_TRIALS = 50

with mlflow.start_run(experiment_id=experiment_id, run_name=run_name_parent, nested=True):
    study = optuna.create_study(direction="maximize")
    study.optimize(
        lambda trial: objective(trial, X_train, y_train, experiment_id),
        n_trials=N_TRIALS,
        callbacks=[champion_callback],
    )

    # Mejores parametros y metrica al run padre
    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_cv_r2", study.best_value)

    mlflow.set_tags(
        tags={
            "project": "Airfoil Self-Noise",
            "optimizer_engine": "optuna",
            "model_family": "sklearn+xgboost",
            "feature_set_version": 1,
            "task": "regression",
        }
    )

    # Reentrenamos el ganador con todo el train y evaluamos en test
    model = build_model_from_params(study.best_params)
    model.fit(X_train, y_train.to_numpy().ravel())

    y_pred = model.predict(X_test)
    test_mae = mean_absolute_error(y_test, y_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    test_r2 = r2_score(y_test, y_pred)

    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_r2", test_r2)

    print(f"Best regressor : {study.best_params['regressor']}")
    print(f"CV R² (best)   : {study.best_value:.4f}")
    print(f"Test R²        : {test_r2:.4f}")
    print(f"Test RMSE      : {test_rmse:.4f}")
    print(f"Test MAE       : {test_mae:.4f}")

    # Logueamos los graficos como artefactos
    mlflow.log_figure(figure=correlation_plot, artifact_file="correlation_plot.png")
    mlflow.log_figure(figure=information_gain_plot, artifact_file="information_gain_plot.png")

    # Logueamos el modelo y capturamos el ModelInfo (MLflow 3.x devuelve model_uri en formato models:/m-...)
    signature = infer_signature(X_train, model.predict(X_train))

    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        signature=signature,
        serialization_format="cloudpickle",
        registered_model_name="airfoil_model_dev",
        metadata={"model_data_version": 1},
    )

    model_uri = model_info.model_uri
    parent_run_id = mlflow.active_run().info.run_id
    print(f"\nmodel_uri    = {model_uri}")
    print(f"parent_run_id = {parent_run_id}")

🏃 View run Trial: 0 at: http://localhost:5001/#/experiments/1/runs/89ab4ac18d5240018039ace5a20b75ad
🧪 View experiment at: http://localhost:5001/#/experiments/1
Initial trial 0 achieved value: 0.8313729747815704


🏃 View run Trial: 1 at: http://localhost:5001/#/experiments/1/runs/c1a73afa9ddc49c2a7812a7adab9c225
🧪 View experiment at: http://localhost:5001/#/experiments/1


🏃 View run Trial: 2 at: http://localhost:5001/#/experiments/1/runs/5bf770cbc8964850bd16ad4a72cb3983
🧪 View experiment at: http://localhost:5001/#/experiments/1


🏃 View run Trial: 3 at: http://localhost:5001/#/experiments/1/runs/9e63f0aa188041a68f82e79649fe62ba
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run Trial: 4 at: http://localhost:5001/#/experiments/1/runs/02472c3728e84f37ab00130058c3b1e1
🧪 View experiment at: http://localhost:5001/#/experiments/1
Trial 4 achieved value: 0.9450542702032054 with  12.0291% improvement
🏃 View run Trial: 5 at: http://localhost:5001/#/experiments/1/runs/afae82a69475427ba556ca86917faa99
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run Trial: 6 at: http://localhost:5001/#/experiments/1/runs/1696ace750864fd198bfbaf435334137
🧪 View experiment at: http://localhost:5001/#/experiments/1


🏃 View run Trial: 7 at: http://localhost:5001/#/experiments/1/runs/d486b089078e45139a0bd431ccc3f808
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run Trial: 8 at: http://localhost:5001/#/experiments/1/runs/d9451c3b63b14013bc1944fd055d511f
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run Trial: 9 at: http://localhost:5001/#/experiments/1/runs/3cc13558f80143cca8c7b5dd01eaec38
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run Trial: 10 at: http://localhost:5001/#/experiments/1/runs/04c6b16901b14ba6b74c0fd77bbf8044
🧪 View experiment at: http://localhost:5001/#/experiments/1


...

2026/04/26 01:51:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Best regressor : XGBoost
CV R² (best)   : 0.9516
Test R²        : 0.9587
Test RMSE      : 1.4386
Test MAE       : 0.9484


Registered model 'airfoil_model_dev' already exists. Creating a new version of this model...
2026/04/26 01:51:32 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: airfoil_model_dev, version 2



model_uri    = models:/m-a839fbdb64864868a48e0692630870ce
parent_run_id = bf5c42bbc32448029fe2d760a2a07245
🏃 View run best_hyperparam_2026/04/26-01:51:18 at: http://localhost:5001/#/experiments/1/runs/bf5c42bbc32448029fe2d760a2a07245
🧪 View experiment at: http://localhost:5001/#/experiments/1


Created version '2' of model 'airfoil_model_dev'.


## Sanity check del modelo guardado

In [7]:
loaded = mlflow.sklearn.load_model(model_uri)

sample = X_test.iloc[[0]]
print("Input :", sample.to_dict(orient="records")[0])
print("Pred  :", float(loaded.predict(sample)[0]))
print("Real  :", float(y_test.iloc[0].values[0]))

Input : {'f': -0.7744157876880715, 'alpha': -1.1597334138855429, 'c': 1.7902351283969078, 'U_infinity': -1.258883204152181, 'delta': -0.5965719598359606}
Pred  : 124.65499114990234
Real  : 125.045


## Registro como modelo productivo (`airfoil_model_prod`, alias `champion`)

La API de prediccion va a cargar el modelo via `client.get_model_version_by_alias("airfoil_model_prod", "champion")`.

In [8]:
from mlflow import MlflowClient

client = MlflowClient()
name = "airfoil_model_prod"
desc = "Regresor para nivel de presion sonora (SSPL) en perfiles de ala NACA 0012"

# Crear modelo registrado solo si no existe
try:
    client.create_registered_model(name=name, description=desc)
except Exception as e:
    print(f"(modelo registrado ya existia: {e})")

tags = {k: str(v) for k, v in model.get_params().items()}
tags["model"] = type(model).__name__
tags["test_r2"] = f"{test_r2:.4f}"
tags["test_rmse"] = f"{test_rmse:.4f}"
tags["test_mae"] = f"{test_mae:.4f}"

result = client.create_model_version(
    name=name,
    source=model_uri,
    run_id=model_info.run_id,
    tags=tags,
)

client.set_registered_model_alias(name, "champion", result.version)

print(f"Registrado {name} v{result.version} con alias champion")

2026/04/26 01:51:32 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: airfoil_model_prod, version 1


Registrado airfoil_model_prod v1 con alias champion


Listo. El modelo champion queda accesible en MLflow para que la API y el DAG de retraining lo consuman.